In [ ]:
# import libraries for data manipulation
import numpy as np
import pandas as pd

# import libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## Importing the Dataset

In [ ]:
# read the data
df = pd.read_csv('austo_automobile.csv')

# returns the first 5 rows
df.head()

In [ ]:
# Shape of the dataset: (rows, columns)
print(f"Number of rows   : {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Datatypes and non-null counts for each column
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe().T

In [ ]:
# Extracting Price statistics
print(f"Minimum Price : ${df['Price'].min():,.0f}")
print(f"Average Price : ${df['Price'].mean():,.0f}")
print(f"Maximum Price : ${df['Price'].max():,.0f}")

In [ ]:
# Check missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Treat missing values:
# - Numerical columns: fill with median (robust to outliers)
# - Categorical columns: fill with mode

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include='object').columns

for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
        print(f"Filled '{col}' with median: {df[col].median()}")

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)
        print(f"Filled '{col}' with mode: {df[col].mode()[0]}")

print("\nMissing values after treatment:")
print(df.isnull().sum().sum())

In [ ]:
# Count of cars by Make type
make_counts = df['Make'].value_counts()
print(make_counts)
print(f"\nNumber of SUV cars: {make_counts.get('SUV', 0)}")

In [ ]:
# ── Numerical Variables: Histogram + Boxplot ──────────────────────────────────
num_cols = ['Age', 'No_of_Dependents', 'Salary', 'Partner_salary', 'Total_salary', 'Price']

fig, axes = plt.subplots(len(num_cols), 2, figsize=(14, 4 * len(num_cols)))
fig.suptitle('Univariate Analysis – Numerical Variables', fontsize=15, y=1.01)

for i, col in enumerate(num_cols):
    # Histogram
    axes[i, 0].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[i, 0].set_title(f'{col} – Distribution')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frequency')

    # Boxplot
    axes[i, 1].boxplot(df[col].dropna(), vert=False, patch_artist=True,
                       boxprops=dict(facecolor='lightblue'))
    axes[i, 1].set_title(f'{col} – Boxplot')
    axes[i, 1].set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical Variables: Countplots ─────────────────────────────────────────
cat_cols = ['Gender', 'Profession', 'Marital_status', 'Education',
            'Personal_loan', 'House_loan', 'Partner_working', 'Make']

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
fig.suptitle('Univariate Analysis – Categorical Variables', fontsize=15)
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, palette='Set2', ax=axes[i])
    axes[i].set_title(f'Count of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=15)
    for p in axes[i].patches:
        axes[i].annotate(f'{int(p.get_height())}',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Filter: Hatchback cars priced above 25000
hatchback_above_25k = df[(df['Make'] == 'Hatchback') & (df['Price'] > 25000)]
print(f"Number of Hatchback cars priced above $25,000: {len(hatchback_above_25k)}")
hatchback_above_25k[['Make', 'Price', 'Salary', 'Profession', 'Education']].head(10)

In [ ]:
# Owners who bought cars priced higher than their individual salary
price_gt_salary = df[df['Price'] > df['Salary']]
print(f"Owners with car price > salary: {len(price_gt_salary)}")

# Among them, how many took a personal loan?
personal_loan_yes = price_gt_salary[price_gt_salary['Personal_loan'] == 'Yes']
print(f"Among them, those with a personal loan: {len(personal_loan_yes)}")

# Percentage
pct = (len(personal_loan_yes) / len(price_gt_salary)) * 100
print(f"Percentage with personal loan: {pct:.1f}%")

In [ ]:
# Visualize: personal loan status among those who spent more than their salary
fig, ax = plt.subplots(figsize=(6, 4))
price_gt_salary['Personal_loan'].value_counts().plot(kind='bar', color=['coral', 'steelblue'],
                                                      edgecolor='white', ax=ax)
ax.set_title('Personal Loan Status – Buyers (Price > Salary)')
ax.set_xlabel('Personal Loan')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9a. Correlation Heatmap ────────────────────────────────────────────────────
plt.figure(figsize=(10, 7))
corr = df[['Age', 'No_of_Dependents', 'Salary', 'Partner_salary', 'Total_salary', 'Price']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, square=True)
plt.title('Correlation Heatmap – Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9b. Pairplot ───────────────────────────────────────────────────────────────
sns.pairplot(df[['Age', 'Salary', 'Total_salary', 'Price', 'Make']], hue='Make',
             palette='Set1', plot_kws={'alpha': 0.5}, diag_kind='kde')
plt.suptitle('Pairplot – Key Numerical Variables by Car Make', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9c. Price vs Categorical Features (Boxplots) ──────────────────────────────
cat_features = ['Make', 'Gender', 'Profession', 'Education', 'Marital_status', 'Personal_loan']

fig, axes = plt.subplots(3, 2, figsize=(15, 14))
fig.suptitle('Price vs Categorical Features', fontsize=15)
axes = axes.flatten()

for i, col in enumerate(cat_features):
    order = df.groupby(col)['Price'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='Price', order=order, palette='Set3', ax=axes[i])
    axes[i].set_title(f'Price by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Price')
    axes[i].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── 9d. Salary vs Price by Make ───────────────────────────────────────────────
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x='Total_salary', y='Price', hue='Make', palette='Set1', alpha=0.6)
plt.title('Total Household Salary vs Car Price by Make', fontsize=13)
plt.xlabel('Total Salary')
plt.ylabel('Car Price')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9e. Car Make distribution by Gender and Profession ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='Make', hue='Gender', palette='Set2', ax=axes[0])
axes[0].set_title('Car Make by Gender')
axes[0].set_xlabel('Make')
axes[0].set_ylabel('Count')

sns.countplot(data=df, x='Make', hue='Profession', palette='Set1', ax=axes[1])
axes[1].set_title('Car Make by Profession')
axes[1].set_xlabel('Make')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# ── 9f. Age distribution by Car Make ─────────────────────────────────────────
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Make', y='Age', palette='pastel',
            order=['Hatchback', 'Sedan', 'SUV'])
plt.title('Age Distribution by Car Make', fontsize=13)
plt.xlabel('Car Make')
plt.ylabel('Age')
plt.tight_layout()
plt.show()

In [ ]:
# Filter: customers with 3 or fewer dependents
few_dependents = df[df['No_of_Dependents'] <= 3]

# Average car price by profession
avg_price_by_profession = few_dependents.groupby('Profession')['Price'].mean().reset_index()
avg_price_by_profession.columns = ['Profession', 'Average_Price']
avg_price_by_profession = avg_price_by_profession.sort_values('Average_Price', ascending=False)
print(avg_price_by_profession)

# Visualization
plt.figure(figsize=(7, 4))
sns.barplot(data=avg_price_by_profession, x='Profession', y='Average_Price',
            palette='muted', edgecolor='black')
plt.title('Average Car Price by Profession\n(Customers with ≤ 3 Dependents)', fontsize=13)
plt.xlabel('Profession')
plt.ylabel('Average Price ($)')
for p in plt.gca().patches:
    plt.gca().annotate(f'${p.get_height():,.0f}',
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Filter: customers with both home loan and personal loan
both_loans = df[(df['House_loan'] == 'Yes') & (df['Personal_loan'] == 'Yes')]
print(f"Total customers with both loans: {len(both_loans)}")

# Price statistics by profession
price_by_profession_loans = both_loans.groupby('Profession')['Price'].describe()
print("\nPrice statistics by Profession (both loans):")
print(price_by_profession_loans)

In [ ]:
# Visualization: Boxplot and Bar plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Car Price by Profession – Customers with Home Loan & Personal Loan', fontsize=13)

# Boxplot
sns.boxplot(data=both_loans, x='Profession', y='Price', palette='Set2', ax=axes[0])
axes[0].set_title('Price Distribution by Profession')
axes[0].set_xlabel('Profession')
axes[0].set_ylabel('Car Price ($)')

# Average price bar chart
avg_price = both_loans.groupby('Profession')['Price'].mean().reset_index()
sns.barplot(data=avg_price, x='Profession', y='Price', palette='Set1',
            edgecolor='black', ax=axes[1])
axes[1].set_title('Average Car Price by Profession')
axes[1].set_xlabel('Profession')
axes[1].set_ylabel('Average Price ($)')
for p in axes[1].patches:
    axes[1].annotate(f'${p.get_height():,.0f}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

**Observations:**
- Even among customers with both a home loan and a personal loan, **Business** profession customers tend to spend more on cars compared to **Salaried** customers.
- This implies that business owners are willing to take on more debt or have higher income to justify purchasing more expensive vehicles despite existing loan obligations.
- Salaried customers with dual loans opt for more affordable cars, reflecting budget constraints.
- This segment represents high-value high-commitment buyers — they are already comfortable with financial products and may be receptive to auto loan offerings.




### Conclusions

**Customer Demographics:**
- The majority of car buyers are **male, married, post-graduate, salaried** individuals aged 25–45.
- Hatchbacks are the most popular car type, followed by Sedans and SUVs.

**Income & Price Relationship:**
- There is a strong positive correlation between **total household salary and car price**.
- SUV buyers have significantly higher salaries than Hatchback buyers.
- A notable segment of customers bought cars priced higher than their individual salary, many of whom took personal loans.

**Profession Influence:**
- **Business owners** consistently spend more on cars than salaried employees across all segments analyzed.
- Even under financial obligation (home + personal loans), business owners purchase more expensive cars.

**Dependency & Loan Behavior:**
- Customers with fewer dependents (≤ 3) tend to spend more on cars.
- Dual-loan holders are an important segment — they are financially active and receptive to additional financial products.










---